# Lesson 16 — Exercises

Run `setup.sql` first, then work through `lesson.ipynb`. Solutions are in `exercises-solutions.ipynb`.

Work the exercises in order. `CHECKPOINT` / `DBCC DROPCLEANBUFFERS` (Exercise 2) are dev-only diagnostics, safe on this learning container. Exercises 4 and 5 mention the actual plan (Ctrl+M) — that graphical view is an SSMS feature.

In [ ]:
USE AdventureWorks2022;
GO

## Exercise 1

Report how many MB of the buffer pool AdventureWorks2022 currently occupies.

**Hint:** `sys.dm_os_buffer_descriptors` grouped by database_id; 8KB per page.

In [ ]:
SELECT DB_NAME(database_id) AS DatabaseName,
       COUNT(*)             AS CachedPages,
       COUNT(*) * 8 / 1024.0  AS CachedMB
FROM sys.dm_os_buffer_descriptors
WHERE database_id = DB_ID('AdventureWorks2022')
GROUP BY database_id;

**Why:** count cached pages for this database_id and convert to MB.

## Exercise 2

Clear the cache (`CHECKPOINT` then `DBCC DROPCLEANBUFFERS`), show that few pages of `lesson16.BigDetail` are cached, scan the table, then show many more are cached.

In [ ]:
CHECKPOINT;
DBCC DROPCLEANBUFFERS;

SELECT COUNT(*) AS CachedBefore
FROM sys.dm_os_buffer_descriptors AS bd
WHERE bd.database_id = DB_ID()
  AND bd.allocation_unit_id IN (
        SELECT au.allocation_unit_id FROM sys.allocation_units au
        JOIN sys.partitions p ON p.partition_id = au.container_id
        WHERE p.object_id = OBJECT_ID('lesson16.BigDetail'));

SELECT COUNT(*) FROM lesson16.BigDetail;   -- forces a scan into cache

SELECT COUNT(*) AS CachedAfter
FROM sys.dm_os_buffer_descriptors AS bd
WHERE bd.database_id = DB_ID()
  AND bd.allocation_unit_id IN (
        SELECT au.allocation_unit_id FROM sys.allocation_units au
        JOIN sys.partitions p ON p.partition_id = au.container_id
        WHERE p.object_id = OBJECT_ID('lesson16.BigDetail'));

**Why:** drop clean buffers, measure, scan, measure again.

## Exercise 3

Report the current Page Life Expectancy in seconds. In a comment, say whether a value of 50 on a busy server would worry you and why.

In [ ]:
SELECT cntr_value AS PLE_Seconds
FROM sys.dm_os_performance_counters
WHERE counter_name = 'Page life expectancy'
  AND object_name LIKE '%Buffer Manager%';

**Why:** read the Buffer Manager PLE counter. A PLE of 50 on a busy server is alarming: pages survive only ~50s in cache, meaning SQL Server is constantly re-reading from disk (memory pressure). On a quiet dev container PLE can be low simply because nothing is keeping it warm — context matters; watch the trend, not one number.

## Exercise 4

Open a transaction that updates some rows of `lesson16.BigDetail` and DO NOT commit yet. In another batch (or just below), show that `sys.databases.log_reuse_wait_desc` for AdventureWorks2022 is ACTIVE_TRANSACTION. Then commit and show it returns to NOTHING.

In [ ]:
BEGIN TRANSACTION;
UPDATE lesson16.BigDetail SET OrderQty = OrderQty WHERE SalesOrderDetailID % 1000 = 0;

SELECT log_reuse_wait_desc FROM sys.databases WHERE name = 'AdventureWorks2022';
-- Expect: ACTIVE_TRANSACTION

COMMIT TRANSACTION;
SELECT log_reuse_wait_desc FROM sys.databases WHERE name = 'AdventureWorks2022';
-- Expect: NOTHING

**Why:** an uncommitted transaction sets log_reuse_wait_desc = ACTIVE_TRANSACTION.

## Exercise 5

Force a tempdb spill on a large sort of `lesson16.BigDetail` using `OPTION (MAX_GRANT_PERCENT = 0.0)`. In a comment, explain what a spill is and one non-hint way to avoid it in real workloads.

In [ ]:
SELECT *
FROM lesson16.BigDetail
ORDER BY ProductID, ModifiedDate, OrderQty, UnitPrice
OPTION (MAX_GRANT_PERCENT = 0.0);

**Why:** starve the memory grant so the sort cannot complete in memory. A spill happens when an operator (sort/hash) is granted too little memory and must write intermediate data to tempdb, doing extra slow I/O. The actual plan flags the Sort with a warning. Non-hint fixes: refresh statistics so the optimizer requests the right grant, reduce the rows/columns being sorted, or add an index that delivers rows already sorted (no sort).